In [1]:
import os
import pickle
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq

FILE_PATH = os.path.dirname(os.path.abspath("."))
os.chdir(FILE_PATH)
load_dotenv()
from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.hybrid import HybridSearch
from src.download_data import download_data
os.chdir(f"{FILE_PATH}/notebooks")

download_data()

CATEGORY = "Appliances"
PROCESSED_DATA_DIR = Path("../data/processed")
with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_product_documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_doc_ids.pkl", "rb") as f:
    doc_ids = pickle.load(f)

import duckdb

PROCESSED_DATA_DIR = Path("../data/processed")
product_data_file = "Appliances_products.parquet"

c2 = duckdb.connect()
products = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{product_data_file}')").df()

/Users/harrisonlee/miniforge3/envs/dsci575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/harrisonlee/Code/ubc-mds/Block 6/DSCI575/DSCI_575_project_hli76_wnsong/src
Review data for Appliances already downloaded
Meta data for Appliances already downloaded
Merged data for Appliances is ready
Products Data for Appliances is ready
Document id for Appliances is ready
Document id for Appliances is ready


In [2]:
print(products.keys())

Index(['parent_asin', 'product_title', 'main_category', 'store', 'price',
       'avg_rating', 'reviews', 'review_titles', 'helpful_votes'],
      dtype='str')


In [3]:
TOP_K = 10



## Create RAG Pipeline using Semantic Retriever

```pseudocode
def pipeline(query):
    retriever = SemanticSearch(Documents)
    retrieved_products = retriever.search(query)
    context = build_docs(retrieved_products)
    prompt = build_prompt(query, context)
    return format_output(llm(prompt))
```

In [4]:
query = "Best container for my food that needs to be cold"

In [5]:
def build_context(results):
    context = ""
    for i, (index, score) in enumerate(results):
        product_asin = doc_ids[index]
        product_context = documents[index]
        # print(f"{i+1}. ({score:.3f}) {product.product_title.values[0]}")
        context += f"""
parent_asin: {product_asin}
{product_context}

"""
    return context

DEFAULT_SYSTEM_PROMPT = """
Instructions:
- You are a helpful Amazon shopping assistant.
- You must answer the question using ONLY the following context (real product reviews with helpful votes and the metadata for the products).
- Always cite the product ASIN when possible.
- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.
"""

def build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT):
    return f"""
{system_prompt}

---------

Context: 
{context}

---------

Question:
{query}

"""

def load_retrievers(documents):
    retrievers = {
        "bm25": BM25Search(documents), 
        "semantic": SemanticSearch(documents)
    }
    retrievers["hybrid"] = HybridSearch(
        bm25=retrievers["bm25"], 
        semantic=retrievers["semantic"], 
        alpha=0.5, 
        top_k_candidates=TOP_K + 100
    )
    return retrievers

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

def RAG_pipeline(retriever, documents, query, llm, top_k=TOP_K):
    retriever = load_retrievers(documents)[retriever]
    if isinstance(retriever, HybridSearch):
        raw_results = retriever.search(query, top_k=top_k)
        results = [(idx, score) for idx, score, _details in raw_results]
    else:
        results = retriever.search(query, top_k=top_k)
    context = build_context(results)
    prompt = build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT)
    response = llm.invoke(prompt).content
    return response

print(RAG_pipeline("hybrid", documents, query, llm))

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5678.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
Based on the reviews, I would recommend the "2-Tier Deviled Egg Container" (ASIN: B0855Q8MZR) or the "Set of 3 Covered Egg Container" (ASIN: B08P797NTS) for storing food that needs to be cold. Both of these containers have received positive reviews for their ability to keep food fresh and cold. 

However, if you're looking for a more general-purpose container, the "Plastic Refrigerator Egg Storage Container" (ASIN: B08P4J57J7) could be a good option. It has a flip-top lid and 12 egg grooves, making it a convenient and space-efficient way to store food in the refrigerator. 

It's also worth considering the "KRIB BLING Compact Refrigerator" (ASIN: B08T682VX8) or the "GiTenvy 15 Quarts Portable Car Cooler" (ASIN: B0B3937GVL) if you need a more portable or compact solution for keeping food cold.


In [7]:
queries = [
    "high-power blender for nut butter and ice under $200", 
    "countertop oven that fits a 9x13 pan and has air fry mode",
    "lightweight cordless vacuum for hardwood, pet hair, and stairs under 5 kg", 
    "convection toaster oven for baking small batches"
]
results = {}
for q in queries:
    results[q] = RAG_pipeline("hybrid", documents, q, llm)

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7320.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10453.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6506.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10578.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done


In [8]:
results

{'high-power blender for nut butter and ice under $200': "I don't know.",
 'countertop oven that fits a 9x13 pan and has air fry mode': 'The B08PN448L7 and B08B3DK2C6 products are wall ovens and the B095YNXNDW is an air fryer toaster oven, but none of them are countertop ovens that fit a 9x13 pan. However, the B095YNXNDW product has a 13.5 Quart capacity which might fit a 9x13 pan. \n\nIt is also worth noting that the B07YZTGYXS and B08DLKXX22 products do not have air fry mode. The B08NVBYTZV product has air fry mode, but it is an induction range, not a countertop oven.\n\nThe B086V54184 product is an electric range with air fry mode, but it is not a countertop oven.\n\nTherefore, the B095YNXNDW product is the closest match to the requirements, but it is an air fryer toaster oven, not a traditional countertop oven. \n\nIt is recommended to check the product dimensions and features to confirm if it fits your needs. The ASIN of the product is B095YNXNDW.',
 'lightweight cordless vacuum f